# Capítulo 6 — Balanço hídrico climatológico operacional

**Curso:** Agrometeorologia Operacional com Python
**Prof. Dr. Fabrício Correia de Oliveira** — UTFPR, Campus Santa Helena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologia/blob/main/curso/06_balanco_hidrico.ipynb)

> Pré-requisito: Capítulos 1 a 5.

---


## 6.1 Motivação

Este é o ponto de virada do curso: até aqui construímos **variáveis** (radiação, temperatura,
graus-dia, umidade, ETo). A partir deste capítulo, essas variáveis viram **decisão** — o
balanço hídrico climatológico de Thornthwaite & Mather (1955) é a peça que alimenta o
zoneamento agroclimático do Capítulo 7 e, depois, o modelo de Machine Learning do Capítulo 8.


## 6.2 Variáveis do balanço hídrico

| Variável | Nome | Significado |
|---|---|---|
| P | Precipitação | Entrada de água (mm) |
| ETP | Evapotransp. Potencial | Demanda atmosférica máxima (mm) — vem do Capítulo 5 |
| NEG.ACUM | Negativo acumulado | Déficit acumulado no solo |
| ARM | Armazenamento | Água atual no solo; 0 ≤ ARM ≤ CAD |
| ALT | Alteração | Variação do armazenamento |
| ETR | Evapotransp. Real | Evapotranspiração efetiva (mm) |
| DEF | Déficit hídrico | ETP − ETR quando ETR < ETP |
| EXC | Excedente hídrico | Água que excede a CAD |
| CAD | Cap. de Água Disponível | Água máxima retida pelo solo (mm) |


## 6.3 Roteiro de cálculo — Thornthwaite & Mather

**Caso 1 — o solo recebe ou mantém água** ($P - ETP \geq 0$):
$$NEG.ACUM = 0 \qquad ARM = \min(ARM_{ant} + (P-ETP),\ CAD)$$
$$ALT = ARM - ARM_{ant} \qquad ETR = ETP \qquad DEF = 0$$
$$EXC = \max(0,\ ARM_{ant} + (P-ETP) - CAD)$$

**Caso 2 — o solo perde água** ($P - ETP < 0$):
$$NEG.ACUM = NEG.ACUM_{ant} + (P-ETP) \qquad ARM = CAD \cdot e^{NEG.ACUM/CAD}$$
$$ALT = ARM - ARM_{ant} \qquad ETR = P - ALT\ (\text{note: } ALT<0,\ \text{logo } ETR>P)$$
$$DEF = ETP - ETR \qquad EXC = 0$$


## 6.4 Do papel ao código

Diferente dos capítulos anteriores (fórmulas independentes por dia), o balanço hídrico é
**recursivo** — cada mês depende do armazenamento (`ARM`) do mês anterior. Vamos implementar
como uma função que recebe séries completas de `P` e `ETP` e devolve o balanço inteiro, mês a
mês, pronto para qualquer extensão de série (não só 12 meses) — essa generalização é o que
torna a função reaproveitável no Capítulo 7.


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def balanco_hidrico_climatologico(P: pd.Series, ETP: pd.Series, CAD: float = 100.0,
                                    ARM_inicial: float | None = None) -> pd.DataFrame:
    """
    Balanço hídrico climatológico de Thornthwaite & Mather.

    P, ETP: séries (mesmo índice) de precipitação e evapotranspiração potencial, em mm,
            para qualquer número de períodos (meses, decêndios etc.).
    CAD: capacidade de água disponível do solo (mm).
    ARM_inicial: armazenamento inicial (mm); default = CAD (solo na capacidade de campo).

    Retorna um DataFrame com todas as variáveis do balanço, um período por linha.
    """
    if ARM_inicial is None:
        ARM_inicial = CAD

    ARM = ARM_inicial
    NEG_ACUM = 0.0
    linhas = []

    for periodo in P.index:
        p, etp = P[periodo], ETP[periodo]
        diff = p - etp

        if diff >= 0:
            NEG_ACUM = 0.0
            ARM_novo = min(ARM + diff, CAD)
            ALT = ARM_novo - ARM
            ETR = etp
            DEF = 0.0
            EXC = max(0.0, ARM + diff - CAD)
        else:
            NEG_ACUM += diff
            ARM_novo = CAD * math.exp(NEG_ACUM / CAD)
            ALT = ARM_novo - ARM
            ETR = p - ALT
            DEF = etp - ETR
            EXC = 0.0

        linhas.append({
            "periodo": periodo, "P": p, "ETP": etp, "P_ETP": diff,
            "NEG_ACUM": round(NEG_ACUM, 2), "ARM": round(ARM_novo, 1), "ALT": round(ALT, 1),
            "ETR": round(ETR, 1), "DEF": round(DEF, 1), "EXC": round(EXC, 1),
        })
        ARM = ARM_novo

    return pd.DataFrame(linhas).set_index("periodo")


## 6.5 Atividade guiada — reproduzindo o balanço anual de Toledo-PR

`CAD = 100 mm`, solo inicia janeiro na capacidade de campo. Os totais esperados (conforme a
apostila): ETR = 968,3 mm, DEF = 1,7 mm, EXC = 308,7 mm.


In [ ]:
meses = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
P   = pd.Series([188,162,128,78,58,38,32,35,80,148,152,178], index=meses)
ETP = pd.Series([130,118,96,72,52,38,36,50,68,90,102,118], index=meses)

bhc_toledo = balanco_hidrico_climatologico(P, ETP, CAD=100.0)
bhc_toledo


In [ ]:
totais = bhc_toledo[["ETR", "DEF", "EXC"]].sum()
print("Totais anuais:")
print(totais.round(1))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(bhc_toledo.index, bhc_toledo["P"], label="Precipitação (P)", marker="o")
ax.plot(bhc_toledo.index, bhc_toledo["ETP"], label="ETP", marker="o")
ax.fill_between(bhc_toledo.index, bhc_toledo["P"], bhc_toledo["ETP"],
                 where=(bhc_toledo["P"] >= bhc_toledo["ETP"]), color="steelblue", alpha=0.2, label="Excedente")
ax.fill_between(bhc_toledo.index, bhc_toledo["P"], bhc_toledo["ETP"],
                 where=(bhc_toledo["P"] < bhc_toledo["ETP"]), color="firebrick", alpha=0.2, label="Déficit")
ax.set_ylabel("mm")
ax.set_title("Balanço hídrico climatológico — Toledo-PR (CAD=100mm)")
ax.legend()
plt.tight_layout()
plt.show()


## 6.6 Desafio

1. Rode o balanço de Toledo-PR para `CAD = 50`, `100` e `150 mm` e compare o déficit e o
   excedente hídrico anual totais. Como a capacidade de água do solo muda a resposta?
2. Usando o `df_clima` do Capítulo 1 (Santa Helena-PR) e a `ETo_HS` calculada no Capítulo 5,
   agregue `P` e `ETP` por mês (`resample("ME").sum()`) e rode o balanço hídrico para essa
   estação real, com `CAD = 100 mm`.


In [ ]:
# Espaço para o desafio — escreva seu código aqui


## 6.7 Checkpoint

Antes de seguir para o **Capítulo 7 — Zoneamento agroclimático via ISNA**, você deve ter:

- [ ] reproduzido o balanço de Toledo-PR com os mesmos totais da apostila;
- [ ] uma função `balanco_hidrico_climatologico(P, ETP, CAD)` genérica, testada, que aceita
      qualquer série de tamanho — não só 12 meses;
- [ ] rodado o balanço para pelo menos um cenário de `CAD` diferente e para dados reais.

Esta função é a base de tudo que vem no Capítulo 7 — o ISNA nada mais é do que essa mesma
lógica, aplicada com a demanda da **cultura** (`ETc`) no lugar da demanda de referência (`ETP`).
